In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Missionaries and Infidels

**Note:** The Python version of this notebook is unfinished.  Only the auxiliary functions that are complete in the Python version have been translated.

In this notebook, numbers are encoded as *bit vectors*, i.e. as lists of propositional formulas.  We use the type `Formula` from the notebook `Propositional-Logic-Parser.ipynb` and the function `normalize` from the notebook `04-CNF.ipynb`.

In [2]:
importNotebook "04-CNF.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb

The function `xor` takes two arguments:
- `a` is a propositional formula,
- `b` is a propositional formula.

It returns the *exclusive or* of `a` and `b`, which is expressed as $a \leftrightarrow \neg b$.

In [3]:
let xor (a: Formula) (b: Formula) : Formula =
    Iff (a, Not b)

The function `halfAdder` takes two arguments:
* `x` is a propositional formula,
* `y` is a propositional formula.

It returns a pair `(carry, sum)` of propositional formulas such that the following holds:
```
2 * carry + sum = x + y
```
The propositional formula `sum` is the *exclusive or* of `x` and `y`, while `carry` is the *logical and*
of `x` and `y`.

In [4]:
let halfAdder (x: Formula) (y: Formula) : Formula * Formula =
    let sum   = xor x y
    let carry = And (x, y)
    (carry, sum)

In [5]:
halfAdder (Var "x") (Var "y")

(And (Var "x", Var "y"), Iff (Var "x", Not (Var "y")))

The function `fullAdder` takes three arguments:
* `x` is a propositional formula,
* `y` is a propositional formula,
* `z` is a propositional formula.

It returns a pair `(carry, sum)` of propositional formulas such that the following holds:
```
2 * carry + sum = x + y + z
```
The propositional formula `sum` is the *exclusive or* of `x`, `y`, and `z`, 
while `carry` is true iff and only if two or more of `x`, `y`, and `z` are true.

In [6]:
let fullAdder (x: Formula) (y: Formula) (z: Formula) : Formula * Formula =
    let sum   = xor x (xor y z)
    let carry = Or (Or (And (x, y), And (x, z)), And (y, z))
    (carry, sum)

In [7]:
fullAdder (Var "x") (Var "y") (Var "z")

(Or (Or (And (Var "x", Var "y"), And (Var "x", Var "z")), And (Var "y", Var "z")), Iff (Var "x", Not (Iff (Var "y", Not (Var "z")))))

The function `adder` takes two arguments:
* `X` is a list of propositional formulas of length $n$,
* `Y` is a list of propositional formulas of the same length $n$.

The function returns a list of propositional formulas `Z` of length $n+1$ such that the following holds:
$$ \sum\limits_{i=0}^n \texttt{Z}[i] \cdot 2^i = \sum\limits_{i=0}^{n-1} \texttt{X}[i] \cdot 2^i + \sum\limits_{i=0}^{n-1} \texttt{Y}[i] \cdot 2^i. $$
In other words: If `X`, `Y`, and `Z` are interpreted as binary numbers, then `Z` is the sum of `X` and `Y`.  Note that the bit with index $0$ is the least significant bit.

**Implementation:** The arrays `C` and `Z` store the carries and the bits of the sum.  `Array.zeroCreate` creates an array whose elements are initialized with a default value, which is `null` for the type `Formula`.  All elements are overwritten before they are used.

In [8]:
let adder (X: Formula list) (Y: Formula list) : Formula list =
    let n = List.length X
    if n <> List.length Y then failwith "adder: lists have different lengths"
    let X, Y = List.toArray X, List.toArray Y
    let C : Formula[] = Array.zeroCreate n
    let Z : Formula[] = Array.zeroCreate (n + 1)
    let c0, z0 = halfAdder X[0] Y[0]
    C[0] <- c0
    Z[0] <- z0
    for i in 1 .. n - 1 do
        let ci, zi = fullAdder X[i] Y[i] C[i-1]
        C[i] <- ci
        Z[i] <- zi
    Z[n] <- C[n-1]
    List.ofArray Z

In [9]:
adder [ Var "x0"; Var "x1" ] [ Var "y0"; Var "y1" ]

[Iff (Var "x0", Not (Var "y0")); Iff (Var "x1", Not (Iff (Var "y1", Not (And (Var "x0", Var "y0"))))); Or (Or (And (Var "x1", Var "y1"), And (Var "x1", And (Var "x0", Var "y0"))), And (Var "y1", And (Var "x0", Var "y0")))]

The function `oneOrTwo` takes one argument:
- `X` is a list of propositional formulas of length 2.

It returns a propositional formula that is true iff `X` interpreted as a binary number either has the value `1` or `2`.

In [10]:
let oneOrTwo (X: Formula list) : Formula =
    xor X[0] X[1]

The function `lessOrEqual` takes two arguments:
* `X` and `Y` are lists of propositional formulas of length 2.

It returns a propositional formula that is true iff we have
$$ 2 \cdot \texttt{X}[1] + \texttt{X}[0] \leq 2 \cdot \texttt{Y}[1] + \texttt{Y}[0]. $$

In [11]:
let lessOrEqual (X: Formula list) (Y: Formula list) : Formula =
    Or (And (Not X[1], Y[1]),
        And (Iff (X[1], Y[1]), Imp (X[0], Y[0])))

The function `disjunction` takes one argument:
- `L` is a non-empty list of formulas.

It returns the disjunction of all formulas in `L`.  The function `conjunction` returns the conjunction of all formulas in `L`.

In [12]:
let rec disjunction (L: Formula list) : Formula =
    match L with
    | [ f ]     -> f
    | f :: rest -> Or (f, disjunction rest)
    | []        -> failwith "disjunction: empty list"

let rec conjunction (L: Formula list) : Formula =
    match L with
    | [ f ]     -> f
    | f :: rest -> And (f, conjunction rest)
    | []        -> failwith "conjunction: empty list"

In [13]:
disjunction [ Var "a"; Var "c"; Var "b" ]

Or (Var "a", Or (Var "c", Var "b"))

In [14]:
conjunction [ Var "a"; Var "c"; Var "b" ]

And (Var "a", And (Var "c", Var "b"))

The function `noProblem` takes two arguments:
* `M` is the number of missionaries on the left shore encoded as a bit vector of length 2,
* `C` is the number of infidels on the left shore encoded as a bit vector of length 2.

It returns a set of clauses that is `true` if there is no problem.  This set of clauses is equivalent to the formula
$$ M = 0 \vee M = 3 \vee M = C. $$

In [15]:
let noProblem (M: Formula list) (C: Formula list) : CNF =
    let L = [ And (Not M[0], Not M[1])                      // M = 0
              And (M[0], M[1])                              // M = 3
              And (Iff (M[0], C[0]), Iff (M[1], C[1])) ]    // M = C
    normalize (disjunction L)

In [16]:
printfn "%s" (cnfToString (noProblem [ Var "M0"; Var "M1" ] [ Var "C0"; Var "C1" ]))

{{C0, M1, ¬M0}, {C1, M0, ¬M1}, {M0, ¬C0, ¬M1}, {M1, ¬C1, ¬M0}}

The function `boatOK` takes two arguments:
* `M` is the number of missionaries on the boat encoded as a bit vector of length 2,
* `C` is the number of infidels on the boat encoded as a bit vector of length 2.

It returns a set of clauses that is `true` if the boat is not empty but there are not more
than two persons on the boat.  This set of clauses is equivalent to the formula
$$ 1 \leq M + C \leq 2. $$

In [17]:
let boatOK (M: Formula list) (C: Formula list) : CNF =
    let L = [ conjunction [ M[0]; Not M[1]; C[0]; Not C[1] ]          // M=1 & C=1
              conjunction [ M[0]; Not M[1]; Not C[0]; Not C[1] ]      // M=1 & C=0
              conjunction [ M[1]; Not M[0]; Not C[0]; Not C[1] ]      // M=2 & C=0
              conjunction [ C[0]; Not C[1]; Not M[0]; Not M[1] ]      // C=1 & M=0
              conjunction [ C[1]; Not C[0]; Not M[0]; Not M[1] ] ]    // C=2 & M=0
    normalize (disjunction L)

In [18]:
printfn "%s" (cnfToString (boatOK [ Var "M0"; Var "M1" ] [ Var "C0"; Var "C1" ]))

{{C0, C1, M0, M1}, {C0, C1, ¬M0, ¬M1}, {C0, M0, ¬C1, ¬M1}, {C0, M1, ¬C1, ¬M0}, {C0, ¬C1, ¬M0}, {C0, ¬C1, ¬M0, ¬M1}, {C0, ¬C1, ¬M1}, {C0, ¬M0, ¬M1}, {C1, M0, ¬C0, ¬M1}, {C1, ¬C0, ¬M0, ¬M1}, {C1, ¬C0, ¬M1}, {C1, ¬M0, ¬M1}, {M0, M1, ¬C0, ¬C1}, {M0, ¬C0, ¬C1}, {M0, ¬C0, ¬C1, ¬M1}, {M0, ¬C0, ¬M1}, {M0, ¬C1, ¬M1}, {M1, ¬C0, ¬C1}, {M1, ¬C0, ¬C1, ¬M0}, {M1, ¬C1, ¬M0}, {¬C0, ¬C1}, {¬C0, ¬C1, ¬M0}, {¬C0, ¬C1, ¬M0, ¬M1}, {¬C0, ¬C1, ¬M1}, {¬C0, ¬M0, ¬M1}, {¬C0, ¬M1}, {¬C1, ¬M0}, {¬C1, ¬M0, ¬M1}, {¬C1, ¬M1}, {¬M0, ¬M1}}